# NCDD Law Library Downloader

Discover records, download available PDFs, and create CSV audit files. Set `TEST_MODE = False` before running the full category.

In [1]:
# # Install required packages and the Playwright browser.
# %pip install requests pandas beautifulsoup4 playwright tqdm
# !playwright install chromium

In [2]:
# All imports in one place.
import asyncio
import csv
import hashlib
import random
import re
from math import ceil
from pathlib import Path
from urllib.parse import parse_qs, urljoin, urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
from tqdm.auto import tqdm

/Users/nak/CADT/CADT_Fourth_Year/info-retrival/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Configuration.
BASE = 'https://library.ncdd.gov.kh'
CATEGORY = 7
PER_PAGE = 12
TEST_MODE = True
TEST_PAGES = 2
DELAY = 2.0
RETRIES = 3
ROOT = Path('ncdd_laws')
PDF_DIR = ROOT / 'pdfs'
ROOT.mkdir(exist_ok=True)
PDF_DIR.mkdir(exist_ok=True)
SESSION = requests.Session()
SESSION.headers.update({'User-Agent': 'LegalRAG academic research downloader (respectful rate limits)'})

In [4]:
# Helper functions.
async def pause():
    await asyncio.sleep(DELAY + random.uniform(0, 0.5))

def listing_url(number):
    return f'{BASE}/product?category={CATEGORY}&page={number}&perpage={PER_PAGE}'

def save_csv(filename, rows):
    rows = list(rows)
    if not rows:
        return
    keys = sorted(set().union(*(row.keys() for row in rows)))
    with (ROOT / filename).open('w', encoding='utf-8-sig', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=keys)
        writer.writeheader()
        writer.writerows(rows)

def valid_pdf(path):
    try:
        with path.open('rb') as file:
            return file.read(5) == b'%PDF-' and path.stat().st_size > 100
    except OSError:
        return False

In [5]:
# Record discovery functions.
async def fetch_page(page, url):
    await page.goto(url, wait_until='domcontentloaded', timeout=60000)
    await page.wait_for_timeout(1200)
    return BeautifulSoup(await page.content(), 'html.parser')

def parse_records(soup, number):
    records = {}
    for link in soup.select('a[href]'):
        href = urljoin(BASE, link.get('href', ''))
        match = re.search(r'/detail/(\d+)(?:[/?#]|$)', href)
        if not match:
            continue
        doc_id, title = match.group(1), link.get_text(' ', strip=True)
        record = {'id': doc_id, 'title': title, 'detail_url': href, 'listing_page': number}
        if doc_id not in records or len(title) > len(records[doc_id]['title']):
            records[doc_id] = record
    return records

def get_total(soup):
    text = soup.get_text(' ', strip=True)
    for pattern in (r'of\s+([\d,]+)\s+records', r'([\d,]+)\s+records', r'ចំនួន\s*([\d,]+)'):
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return int(match.group(1).replace(',', ''))
    return None

def get_last_page(soup):
    numbers = []
    for link in soup.select('a[href]'):
        query = parse_qs(urlparse(urljoin(BASE, link.get('href', ''))).query)
        if 'page' in query and query['page'][0].isdigit():
            numbers.append(int(query['page'][0]))
    return max(numbers) if numbers else None

async def discover_records():
    records, failed_pages = {}, []
    async with async_playwright() as playwright:
        browser = await playwright.chromium.launch(headless=True)
        page = await browser.new_page()
        try:
            first = await fetch_page(page, listing_url(1))
            expected, last = get_total(first), get_last_page(first)
            if expected is not None:
                total_pages = ceil(expected / PER_PAGE)
            elif last is not None:
                total_pages = last
            else:
                raise RuntimeError('Could not determine page count; inspect page HTML.')
            if TEST_MODE:
                total_pages = min(TEST_PAGES, total_pages)
            print('Reported records:', expected, '| Pages to scan:', total_pages)
            for number in tqdm(range(1, total_pages + 1), desc='Listing pages'):
                success = False
                for attempt in range(RETRIES):
                    try:
                        soup = first if number == 1 else await fetch_page(page, listing_url(number))
                        found = parse_records(soup, number)
                        if not found:
                            raise RuntimeError('No detail links found; page may have changed')
                        records.update(found)
                        success = True
                        break
                    except Exception as error:
                        print(f'Page {number}, attempt {attempt + 1}: {error}')
                        await pause()
                if not success:
                    failed_pages.append(number)
                save_csv('discovered.csv', records.values())
                await pause()
        finally:
            await browser.close()
    return records, failed_pages, expected

In [6]:
# PDF detection and download functions.
def candidates_from_html(soup, detail_url):
    candidates = []
    for tag in soup.select('[href], [src], [data]'):
        for attribute in ('href', 'src', 'data'):
            raw = tag.get(attribute)
            if not raw or raw.startswith(('data:', 'blob:', 'javascript:')):
                continue
            url = urljoin(detail_url, raw)
            parsed = urlparse(url)
            label = tag.get_text(' ', strip=True).lower()
            if parsed.scheme in ('http', 'https') and ('.pdf' in parsed.path.lower() or parsed.hostname == 'lib.ncdd.gov.kh' or any(word in label for word in ('download', 'ទាញយក', 'pdf'))):
                candidates.append(url)
    return list(dict.fromkeys(candidates))

def download_pdf(url, destination, referer):
    temporary = destination.with_suffix('.part')
    try:
        with SESSION.get(url, headers={'Referer': referer}, timeout=(20, 120), stream=True) as response:
            response.raise_for_status()
            if 'text/html' in response.headers.get('Content-Type', '').lower():
                raise ValueError('Received HTML, not PDF')
            with temporary.open('wb') as file:
                for chunk in response.iter_content(65536):
                    if chunk:
                        file.write(chunk)
        if not valid_pdf(temporary):
            raise ValueError('Invalid PDF signature or empty file')
        temporary.replace(destination)
        return hashlib.sha256(destination.read_bytes()).hexdigest()
    finally:
        temporary.unlink(missing_ok=True)

async def process_record(record, page):
    destination = PDF_DIR / f"{record['id']}.pdf"
    if valid_pdf(destination):
        record.update(status='downloaded', file=str(destination), note='already present')
        return
    record.pop('error', None)
    observed = []
    def on_response(response):
        try:
            if 'application/pdf' in response.headers.get('content-type', '').lower():
                observed.append(response.url)
        except Exception:
            pass
    page.on('response', on_response)
    try:
        soup = await fetch_page(page, record['detail_url'])
        candidates = list(dict.fromkeys(candidates_from_html(soup, record['detail_url']) + observed))
    finally:
        page.remove_listener('response', on_response)
    record['candidate_urls'] = ' | '.join(candidates)
    if not candidates:
        record['status'] = 'no_pdf_link'
        return
    errors = []
    for url in candidates:
        for attempt in range(RETRIES):
            try:
                digest = await asyncio.to_thread(download_pdf, url, destination, record['detail_url'])
                record.update(status='downloaded', file=str(destination), pdf_url=url, sha256=digest)
                return
            except Exception as error:
                errors.append(f'{url}: {error}')
                await pause()
    record.update(status='download_failed', error='; '.join(errors)[-2000:])

async def download_all_records(records):
    if not records:
        raise RuntimeError('No records were discovered; fix discovery first.')
    audit = {key: dict(value) for key, value in records.items()}
    async with async_playwright() as playwright:
        browser = await playwright.chromium.launch(headless=True)
        page = await browser.new_page()
        try:
            for record in tqdm(audit.values(), desc='Documents'):
                for attempt in range(RETRIES):
                    try:
                        await process_record(record, page)
                        break
                    except Exception as error:
                        record.update(status='detail_failed', error=str(error))
                        await pause()
                save_csv('audit.csv', audit.values())
                await pause()
        finally:
            await browser.close()
    return audit

In [7]:
# One function for the complete workflow.
async def run_downloader():
    records, failed_pages, expected = await discover_records()
    print('Unique records:', len(records), '| Failed listing pages:', failed_pages)
    if not TEST_MODE and expected is not None and len(records) != expected:
        print('WARNING: discovered count differs from reported count.')
    audit = await download_all_records(records)
    report = pd.DataFrame(audit.values())
    if report.empty:
        raise RuntimeError('No audit records exist; check discovery and downloading.')
    report.to_csv(ROOT / 'audit.csv', index=False, encoding='utf-8-sig')
    missing = report[report['status'] != 'downloaded']
    missing.to_csv(ROOT / 'unresolved.csv', index=False, encoding='utf-8-sig')
    valid_count = sum(valid_pdf(PDF_DIR / f'{doc_id}.pdf') for doc_id in records)
    complete_discovery = not failed_pages and (TEST_MODE or expected is None or len(records) == expected)
    complete_download = valid_count == len(records)
    print(pd.Series([record.get('status', 'unknown') for record in audit.values()]).value_counts())
    print(f'Discovered: {len(records)} | Valid PDFs: {valid_count} | Unresolved: {len(missing)}')
    print('Full category complete:', bool(not TEST_MODE and complete_discovery and complete_download))
    print('Audit:', ROOT / 'audit.csv', '| Unresolved:', ROOT / 'unresolved.csv')
    return records, audit

In [8]:
# Run the complete downloader.
records, audit = await run_downloader()

Reported records: None | Pages to scan: 2


Listing pages: 100%|██████████| 2/2 [00:06<00:00,  3.22s/it]


Unique records: 24 | Failed listing pages: []


Documents: 100%|██████████| 24/24 [05:03<00:00, 12.65s/it]

downloaded    24
Name: count, dtype: int64
Discovered: 24 | Valid PDFs: 24 | Unresolved: 0
Full category complete: False
Audit: ncdd_laws/audit.csv | Unresolved: ncdd_laws/unresolved.csv


In [9]:
# Review unresolved records after the run.
report = pd.DataFrame(audit.values())
missing = report[report['status'] != 'downloaded']
display(missing[['id', 'title', 'status']].head(30) if len(missing) else report[['id', 'title', 'status']].head())

,id,title,status
0,17332,ច្បាប់ស្តីពីការចាត់ចែងជាបណ្តោះអាសន្ននូវវត្ថុចា...,downloaded
1,17319,ច្បាប់ស្តីពី របបសន្តិសុខសង្គម,downloaded
2,17260,ច្បាប់ ស្ដីពីការគ្រប់គ្រងរដ្ឋបាល ឃុំ សង្កាត់,downloaded
3,17204,ព្រះរាជក្រម នស/រកម/០៧២៦/០១០ ចុះថ្ងៃទី១០​ ខែកក្...,downloaded
4,17192,ច្បាប់_ស្ដីពី_គណនេយ្យ_និងសវនកម្ម,downloaded
